# PSO-AFW-CIL Quick Test — 3 Kịch bản trên 1 Tập dữ liệu nhỏ

Notebook test nhỏ gọn, dùng **haberman** cho cả 3 kịch bản:

| KB | Mô tả |
|---|---|
| **KB1** | W-SVM vs AFW-CIL-fixed vs GridSearch-AFW-CIL vs PSO-AFW-CIL |
| **KB2** | 6-cấu hình hybrid pipeline (± BL-SMOTE) |
| **KB3** | Stress-test IR bằng `change_rate_data` (7 mức IR từ 20% → 2%) |

Mỗi lần chạy tạo thư mục `./Experiment/Test_DDMMYYYY_HHMMSS/` riêng.  
Các file CSV được lưu dần sau mỗi fold (không mất khi crash).


## Cell 1 — Imports & Môi trường

In [ ]:
import sys, os, math, csv, time, tracemalloc, glob
import numpy as np
import pandas as pd
from datetime import datetime

# ── sys.path ──────────────────────────────────────────────────
parent_dir = os.path.abspath('..')
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)
print(f"sys.path += {parent_dir}")

# ── sklearn ───────────────────────────────────────────────────
from sklearn.svm import SVC
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import (confusion_matrix, roc_auc_score, f1_score,
                              accuracy_score, classification_report)
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split as tts

# ── imbalanced-learn ──────────────────────────────────────────
from imblearn.over_sampling import BorderlineSMOTE

# ── project modules ───────────────────────────────────────────
from wsvm.application import Wsvm
from svm.application import Svm
from fuzzy.weight import fuzzy
from data.common.change_rate_data import change_rate_data

# ── Matplotlib ────────────────────────────────────────────────
import matplotlib.pyplot as plt

# ── Thư mục lưu cho lần chạy này ─────────────────────────────
_RUN_TS  = datetime.now().strftime("%d%m%Y_%H%M%S")
RUN_DIR  = f"./Experiment/Test_{_RUN_TS}"
LOG_DIR  = os.path.join(RUN_DIR, "logs")
os.makedirs(RUN_DIR,  exist_ok=True)
os.makedirs(LOG_DIR,  exist_ok=True)

print(f"✔ Imports OK")
print(f"✔ Thư mục lần chạy: {RUN_DIR}")


## Cell 2 — Tải & Tiền xử lý Dataset

In [ ]:
# ── Cấu hình dataset (chỉ cần đổi ở đây để thử dataset khác) ─
DATASET_NAME = "haberman"       # << đổi tên dataset tại đây
DATASET_FILE = "../Processing_Data/dataset/haberman.csv"
LABEL_COL    = "class"
LABEL_MAP    = {2: 1.0, 1: -1.0}   # lớp thiểu số (2) → +1
DROP_COLS    = []
CAT_COLS     = []
IMPUTE       = False

# ── Tham số chạy nhanh (nhỏ để test) ─────────────────────────
N_SPLITS      = 5       # số fold CV
N_REPEATS     = 2       # số lần lặp (tăng lên 10 cho thực nghiệm thật)
PSO_PARTICLES = 5       # số hạt PSO
PSO_ITERS     = 5       # số vòng PSO (tăng lên 20 cho thực nghiệm thật)
C             = 100
T_INNER       = 3       # số vòng LFB inner (tăng lên 5)
NAMEMETHOD    = "distance_center_own_opposite_tam"
NAMEFUNCTION  = "func_own_opp_new"
BOUNDS        = [(3, 11), (0.01, 0.5), (0.01, 0.99), (0.01, 0.5), (0.01, 0.99)]

# ── Đọc file ─────────────────────────────────────────────────
df_raw = pd.read_csv(DATASET_FILE)
print(f"Raw shape : {df_raw.shape}")
print(f"Columns   : {df_raw.columns.tolist()}")

# Label mapping
df_raw[LABEL_COL] = df_raw[LABEL_COL].map(LABEL_MAP)
df_raw = df_raw.dropna(subset=[LABEL_COL])

y_all = df_raw[LABEL_COL].values.astype(float)
X_df  = df_raw.drop(columns=[LABEL_COL] + DROP_COLS)

# Categorical encoding
if CAT_COLS:
    cat_idx = [X_df.columns.get_loc(c) for c in CAT_COLS]
    ct = ColumnTransformer(
        [('ohe', OneHotEncoder(sparse_output=False, handle_unknown='ignore'), cat_idx)],
        remainder='passthrough')
    X_all = np.array(ct.fit_transform(X_df.values), dtype=float)
else:
    X_all = X_df.values.astype(float)

# Imputation
if IMPUTE:
    X_all = SimpleImputer(strategy='mean').fit_transform(X_all)

# Class distribution
pos_count = int(np.sum(y_all == 1.0))
neg_count = int(np.sum(y_all == -1.0))
ir_natural = pos_count / len(y_all)
print(f"\nTổng mẫu  : {len(y_all)}")
print(f"Lớp +1    : {pos_count}  ({100*pos_count/len(y_all):.1f}%)")
print(f"Lớp -1    : {neg_count}  ({100*neg_count/len(y_all):.1f}%)")
print(f"IR (pos)  : {ir_natural:.4f}")
ir_imb = neg_count / max(pos_count, 1)
print(f"IR (neg/pos): {ir_imb:.4f}")

# ── Lưu checkpoint dataset đã làm sạch ───────────────────────
df_clean = pd.DataFrame(X_all)
df_clean['label'] = y_all
checkpoint_path = os.path.join(RUN_DIR, f"dataset_cleaned_{DATASET_NAME}.csv")
df_clean.to_csv(checkpoint_path, index=False)
print(f"\n✔ Đã lưu dataset sạch: {checkpoint_path}")


## Cell 3 — Wrappers Phân loại Cơ sở & Hàm Tiện ích

In [ ]:
# ── Wrappers ─────────────────────────────────────────────────
def svm_lib(X_train, y_train, X_test):
    return SVC(probability=True, kernel='linear').fit(X_train, y_train).predict(X_test)

def wsvm(C, X_train, y_train, X_test, distribution_weight=None):
    m = Wsvm(C, distribution_weight)
    m.fit(X_train, y_train)
    return m.predict(X_test)

def svm(C, X_train, y_train, X_test):
    m = Svm(C)
    m.fit(X_train, y_train)
    return m.predict(X_test)


# ── Tomek Links ───────────────────────────────────────────────
def is_tomek(X, y, class_type):
    nn = NearestNeighbors(n_neighbors=2)
    nn.fit(X)
    _, nbr = nn.kneighbors(X)
    nn_idx = nbr[:, 1]
    links = np.zeros(len(y), dtype=bool)
    excluded = [c for c in np.unique(y) if c not in class_type]
    X_dangxet, X_tl = [], []
    for i, tgt in enumerate(y):
        if tgt in excluded:
            continue
        if y[nn_idx[i]] != tgt and nn_idx[nn_idx[i]] == i:
            X_tl.append(i)
            X_dangxet.append(nn_idx[i])
            links[i] = True
    return links, X_dangxet, X_tl


# ── G-mean (safe 2×2 CM) ─────────────────────────────────────
def Gmean(y_test, y_pred):
    cm = confusion_matrix(y_test, y_pred, labels=[-1.0, 1.0])
    TN, FP, FN, TP = cm.ravel()
    SE = TP / (TP + FN) if (TP + FN) > 0 else 0.0
    SP = TN / (TN + FP) if (TN + FP) > 0 else 0.0
    return SP, SE, math.sqrt(SE * SP)


# ── CSV helpers ───────────────────────────────────────────────
CSV_HEADER = ['Times', 'Fold', 'T', 'Name Method', 'Name Function',
              'SP', 'SE', 'Gmean', 'F1 Score', 'Accuracy', 'AUC', 'Confusion Matrix']

def append_csv_row(filepath, row):
    write_header = not os.path.exists(filepath)
    with open(filepath, 'a', encoding='UTF8', newline='') as f:
        w = csv.writer(f)
        if write_header:
            w.writerow(CSV_HEADER)
        w.writerow(row)

def make_row(times, fold, t_iter, namemethod, namefunction,
             sp, se, gm, f1, acc, auc, cm_str):
    return [times, fold, t_iter, namemethod, namefunction,
            round(float(sp), 4), round(float(se), 4), round(float(gm), 4),
            round(float(f1), 4), round(float(acc), 4), round(float(auc), 4),
            cm_str]


# ── Ghi metrics vào file text ─────────────────────────────────
def metr_text(f, X_train, y_test, pred, sp, se, gm):
    f.write(f"\nTrain={len(X_train)} | Test={len(y_test)}\n")
    f.write(classification_report(y_test, pred, zero_division=0))
    f.write(f"SP={sp:.4f} | SE={se:.4f} | Gmean={gm:.4f} | "
            f"F1={f1_score(y_test, pred, zero_division=0):.4f} | "
            f"Acc={accuracy_score(y_test, pred):.4f} | "
            f"AUC={roc_auc_score(y_test, pred):.4f}\n")


print("✔ Wrappers & utility functions OK")


## Cell 4 — Fuzzy Weight (CalFW) + AFW-CIL Core + LFB + PSO + GridSearch

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  FUZZY WEIGHT – CalFW                                    ║
# ╚══════════════════════════════════════════════════════════╝
def compute_weight(X, y,
                   name_method="distance_center_own_opposite_tam",
                   name_function="func_own_opp_new",
                   beta=None, C=None, gamma=None, u=None, sigma=None):
    method   = fuzzy.method()
    function = fuzzy.function()
    pos_index = np.where(y == 1)[0]
    neg_index = np.where(y == -1)[0]
    if name_method == "own_class_center":
        d = method.own_class_center(X, y)
    elif name_method == "estimated_hyper_lin":
        d = method.estimated_hyper_lin(X, y)
    elif name_method == "own_class_center_opposite":
        d = method.own_class_center_opposite(X, y)
    elif name_method == "actual_hyper_lin":
        d = method.actual_hyper_lin(X, y, C=C, gamma=gamma)
    elif name_method == "own_class_center_divided":
        d = method.own_class_center_divided(X, y)
    elif name_method == "distance_center_own_opposite_tam":
        d_own, d_opp, d_tam = method.distance_center_own_opposite_tam(X, y)
    else:
        raise ValueError(f"Unknown name_method: {name_method}")

    if name_function == "lin":
        W = function.lin(d)
    elif name_function == "exp":
        W = function.exp(d, beta)
    elif name_function == "lin_center_own":
        W = function.lin_center_own(d, pos_index, neg_index)
    elif name_function == "gau":
        W = function.gau(d, u, sigma)
    elif name_function == "func_own_opp_new":
        W = function.func_own_opp_new(d_own, d_opp, pos_index, neg_index, d_tam)
    else:
        raise ValueError(f"Unknown name_function: {name_function}")

    W = np.array(W)
    r_neg = len(pos_index) / max(len(neg_index), 1)
    m = np.zeros(len(y))
    m[pos_index] = W[pos_index] * 1.0
    m[neg_index] = W[neg_index] * r_neg
    return m


def fuzzy_weight(beta_center, beta_estimate, beta_actual,
                 X_train, y_train, namemethod, namefunction):
    beta_map = {
        ("own_class_center_opposite", "exp"): beta_center,
        ("own_class_center",          "exp"): beta_estimate,
        ("own_class_center_divided",  "exp"): beta_estimate,
        ("estimated_hyper_lin",       "exp"): beta_estimate,
        ("actual_hyper_lin",          "exp"): beta_actual,
    }
    beta = beta_map.get((namemethod, namefunction), None)
    return compute_weight(X_train, y_train,
                          name_method=namemethod, name_function=namefunction,
                          beta=beta)


print("✔ Fuzzy weight OK")


In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  AFW-CIL CORE – Fixed & Parametric                       ║
# ╚══════════════════════════════════════════════════════════╝
FIXED_RATE = 1.2

def data_tomelinks_final(C, ind_posX, ind_negX, weight,
                          X_test, y_test, X_train, y_train, K_neighbors):
    new_W   = np.copy(weight)
    neg_idx = np.where(y_train == -1)[0]
    clf = Wsvm(C, new_W)
    clf.fit(X_train, y_train)
    y_predict = clf.predict(X_test)
    _, se, gm = Gmean(y_test, y_predict)

    nn2 = NearestNeighbors(n_neighbors=K_neighbors)
    nn2.fit(X_train)

    neg_pred = clf.predict(X_train[neg_idx])
    wrong    = np.where(neg_pred != -1.0)[0]
    new_W[neg_idx[wrong]] /= FIXED_RATE

    ind_nn, y_nn = [], []
    for ind, i in enumerate(ind_posX):
        pred = clf.predict(X_train[i:i+1])
        if pred == -1.0:
            ind_nn.append(ind)
            for j in nn2.kneighbors(X_train[i:i+1])[1].flatten():
                y_nn.append(y_train[j])
        else:
            new_W[ind_negX[ind]] /= FIXED_RATE
            new_W[i]             *= FIXED_RATE

    if len(y_nn) > 0:
        y_nn = np.array_split(np.array(y_nn), max(1, len(y_nn) // K_neighbors))
        for ind_k, _ in enumerate(y_nn):
            if 1 not in y_nn[ind_k][1:]:
                new_W[ind_posX[ind_nn[ind_k]]] /= FIXED_RATE
            else:
                new_W[ind_negX[ind_nn[ind_k]]] /= FIXED_RATE
                new_W[ind_posX[ind_nn[ind_k]]] *= FIXED_RATE
    return new_W, gm, se


def data_tomelinks_parametric(C, ind_posX, ind_negX, weight,
                               X_test, y_test, X_train, y_train,
                               K_neighbors, sigma_1, sigma_2, sigma_3, sigma_4):
    new_W   = np.copy(weight)
    neg_idx = np.where(y_train == -1)[0]
    clf = Wsvm(C, new_W)
    clf.fit(X_train, y_train)
    y_predict = clf.predict(X_test)
    _, se, gm = Gmean(y_test, y_predict)

    nn2 = NearestNeighbors(n_neighbors=K_neighbors)
    nn2.fit(X_train)

    neg_pred = clf.predict(X_train[neg_idx])
    wrong    = np.where(neg_pred != -1.0)[0]
    new_W[neg_idx[wrong]] *= sigma_2

    ind_nn, y_nn = [], []
    for ind, i in enumerate(ind_posX):
        pred = clf.predict(X_train[i:i+1])
        if pred == -1.0:
            ind_nn.append(ind)
            for j in nn2.kneighbors(X_train[i:i+1])[1].flatten():
                y_nn.append(y_train[j])
        else:
            new_W[ind_negX[ind]] *= (1 - sigma_1)
            new_W[i]             *= (1 + sigma_1)

    if len(y_nn) > 0:
        y_nn = np.array_split(np.array(y_nn), max(1, len(y_nn) // K_neighbors))
        for ind_k, _ in enumerate(y_nn):
            if 1 not in y_nn[ind_k][1:]:
                new_W[ind_posX[ind_nn[ind_k]]] *= sigma_4
            else:
                new_W[ind_negX[ind_nn[ind_k]]] *= (1 - sigma_3)
                new_W[ind_posX[ind_nn[ind_k]]] *= (1 + sigma_3)
    return new_W, gm, se


# ── LFB Loops ─────────────────────────────────────────────────
def lfb_fixed(C, ind_posX, ind_negX, weight, T, X_test, y_test, X_train, y_train,
              K_neighbors=6):
    gmax, best_w = 0.0, np.copy(weight)
    cur_w = np.copy(weight)
    history = []
    for _ in range(T):
        cur_w, gm, _ = data_tomelinks_final(
            C, ind_posX, ind_negX, cur_w, X_test, y_test, X_train, y_train, K_neighbors)
        history.append(gm)
        if gm > gmax:
            gmax   = gm
            best_w = np.copy(cur_w)
    return best_w, gmax, history


def lfb_pso(C, ind_posX, ind_negX, weight, T,
            X_test, y_test, X_train, y_train,
            K_neighbors, s1, s2, s3, s4):
    gmax, best_w = 0.0, np.copy(weight)
    cur_w = np.copy(weight)
    for _ in range(T):
        cur_w, gm, _ = data_tomelinks_parametric(
            C, ind_posX, ind_negX, cur_w,
            X_test, y_test, X_train, y_train,
            K_neighbors, s1, s2, s3, s4)
        if gm > gmax:
            gmax   = gm
            best_w = np.copy(cur_w)
    return best_w, gmax


print("✔ AFW-CIL (fixed & parametric) + LFB OK")


In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  PSO OPTIMIZER                                            ║
# ╚══════════════════════════════════════════════════════════╝
class PSO_AFWCIL:
    def __init__(self, num_particles, iters, bounds, C, T,
                 X_train, y_train, X_test, y_test,
                 namemethod, namefunction):
        self.num_particles = num_particles
        self.iters         = iters
        self.bounds        = bounds
        self.C             = C
        self.T             = T
        self.X_train       = X_train
        self.y_train       = y_train
        self.X_test        = X_test
        self.y_test        = y_test
        self.namemethod    = namemethod
        self.namefunction  = namefunction
        _, self.ind_posX, self.ind_negX = is_tomek(
            X_train, y_train, class_type=[-1.0])
        self.init_weight = fuzzy_weight(
            0.5, 0.8, 0.2, X_train, y_train, namemethod, namefunction)
        self.dim = len(bounds)

    def fitness_function(self, particle):
        K = max(1, int(round(float(particle[0]))))
        s1, s2, s3, s4 = particle[1], particle[2], particle[3], particle[4]
        _, gmax = lfb_pso(
            self.C, self.ind_posX, self.ind_negX, self.init_weight, self.T,
            self.X_test, self.y_test, self.X_train, self.y_train,
            K, s1, s2, s3, s4)
        return gmax

    def optimize(self):
        n, dim = self.num_particles, self.dim
        pos = np.array([[self.bounds[d][0] + np.random.rand()
                         * (self.bounds[d][1] - self.bounds[d][0])
                         for d in range(dim)] for _ in range(n)], dtype=float)
        vel = np.zeros_like(pos)
        pbest_pos = pos.copy()
        pbest_val = np.array([self.fitness_function(pos[i]) for i in range(n)])
        gbest_idx = int(np.argmax(pbest_val))
        gbest_pos = pbest_pos[gbest_idx].copy()
        gbest_val = float(pbest_val[gbest_idx])
        convergence = []
        w, c1, c2 = 0.7, 1.5, 1.5
        for _ in range(self.iters):
            for i in range(n):
                r1, r2 = np.random.rand(dim), np.random.rand(dim)
                vel[i] = (w * vel[i]
                          + c1 * r1 * (pbest_pos[i] - pos[i])
                          + c2 * r2 * (gbest_pos    - pos[i]))
                pos[i] = np.clip(pos[i] + vel[i],
                                 [b[0] for b in self.bounds],
                                 [b[1] for b in self.bounds])
                fval = self.fitness_function(pos[i])
                if fval > pbest_val[i]:
                    pbest_val[i] = fval
                    pbest_pos[i] = pos[i].copy()
                if fval > gbest_val:
                    gbest_val = fval
                    gbest_pos = pos[i].copy()
            convergence.append(gbest_val)
        return gbest_pos, gbest_val, convergence


# ── GridSearch baseline ───────────────────────────────────────
def grid_search_afwcil(C, T, X_train, y_train, X_test, y_test,
                        namemethod, namefunction,
                        K_grid=None, sigma_grid=None):
    if K_grid     is None: K_grid     = [3, 5, 7, 9, 11]
    if sigma_grid is None: sigma_grid = [0.1, 0.2, 0.3, 0.5]
    _, ind_posX, ind_negX = is_tomek(X_train, y_train, class_type=[-1.0])
    init_w = fuzzy_weight(0.5, 0.8, 0.2, X_train, y_train, namemethod, namefunction)
    best_gmean, best_params = -1.0, {}
    t0 = time.time()
    for K in K_grid:
        for sigma in sigma_grid:
            _, gmax = lfb_pso(C, ind_posX, ind_negX, init_w, T,
                               X_test, y_test, X_train, y_train,
                               K, sigma, sigma, sigma, sigma)
            if gmax > best_gmean:
                best_gmean  = gmax
                best_params = {"K": K, "sigma": sigma}
    return best_params, best_gmean, round(time.time() - t0, 4)


print("✔ PSO_AFWCIL class + grid_search_afwcil OK")


---
## Kịch bản 1 (KB1) — So sánh 4 phương pháp trên cùng dataset

| # | Phương pháp | Ghi chú |
|---|---|---|
| 1 | **W-SVM** | Baseline |
| 2 | **AFW-CIL (σ=1.2 cố định)** | K=6 |
| 3 | **GridSearch-AFW-CIL** | Duyệt K×σ |
| 4 | **PSO-AFW-CIL** | PSO tối ưu {K, σ₁…σ₄} |

CSV lưu vào `{RUN_DIR}/KB1_{dataset}_{ts}.csv`

In [ ]:
def run_kb1(X_all, y_all, run_dir, dataset_name=DATASET_NAME):
    """
    KB1: So sánh W-SVM / AFW-CIL-fixed / GridSearch / PSO-AFW-CIL.
    Trả về: (csv_path, df_summary)
    """
    ts       = datetime.now().strftime("%d%m%Y_%H%M%S")
    csv_path = os.path.join(run_dir, f"KB1_{dataset_name}_{ts}.csv")

    print(f"\n{'='*60}")
    print(f"KB1 | Dataset: {dataset_name} | {N_REPEATS}×{N_SPLITS}-fold CV")
    print(f"Output: {csv_path}")
    print(f"{'='*60}")

    for rep in range(1, N_REPEATS + 1):
        skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=rep * 7)
        for fold, (tr_idx, te_idx) in enumerate(skf.split(X_all, y_all), 1):
            X_tr_raw, X_te_raw = X_all[tr_idx], X_all[te_idx]
            y_tr, y_te         = y_all[tr_idx], y_all[te_idx]

            sc   = StandardScaler()
            X_tr = sc.fit_transform(X_tr_raw)
            X_te = sc.transform(X_te_raw)

            # ── 1. W-SVM ──────────────────────────────────────────
            t0   = time.time()
            pred = wsvm(C, X_tr, y_tr, X_te, np.ones(len(y_tr)))
            sp, se, gm = Gmean(y_te, pred)
            f1  = f1_score(y_te, pred, pos_label=1, zero_division=0)
            acc = accuracy_score(y_te, pred)
            auc = roc_auc_score(y_te, pred)
            append_csv_row(csv_path, make_row(
                rep, fold, round(time.time()-t0, 4),
                "W-SVM", "baseline",
                sp, se, gm, f1, acc, auc,
                str(confusion_matrix(y_te, pred, labels=[-1., 1.]).tolist())))

            # ── 2. AFW-CIL (σ=1.2) ───────────────────────────────
            t0 = time.time()
            _, posX, negX = is_tomek(X_tr, y_tr, class_type=[-1.0])
            w_init = fuzzy_weight(0.5, 0.8, 0.2, X_tr, y_tr, NAMEMETHOD, NAMEFUNCTION)
            best_w, _, _ = lfb_fixed(C, posX, negX, w_init, T_INNER, X_te, y_te, X_tr, y_tr)
            pred = wsvm(C, X_tr, y_tr, X_te, best_w)
            sp, se, gm = Gmean(y_te, pred)
            f1  = f1_score(y_te, pred, pos_label=1, zero_division=0)
            acc = accuracy_score(y_te, pred)
            auc = roc_auc_score(y_te, pred)
            append_csv_row(csv_path, make_row(
                rep, fold, round(time.time()-t0, 4),
                "AFW-CIL_fixed", "fixed_sigma_1.2",
                sp, se, gm, f1, acc, auc,
                str(confusion_matrix(y_te, pred, labels=[-1., 1.]).tolist())))

            # ── 3. GridSearch-AFW-CIL ────────────────────────────
            t0 = time.time()
            best_p, _, _ = grid_search_afwcil(
                C, T_INNER, X_tr, y_tr, X_te, y_te, NAMEMETHOD, NAMEFUNCTION,
                K_grid=[3, 5, 7, 9], sigma_grid=[0.1, 0.3, 0.5])
            _, posX, negX = is_tomek(X_tr, y_tr, class_type=[-1.0])
            w_init = fuzzy_weight(0.5, 0.8, 0.2, X_tr, y_tr, NAMEMETHOD, NAMEFUNCTION)
            s = best_p["sigma"]
            best_w, _ = lfb_pso(C, posX, negX, w_init, T_INNER,
                                  X_te, y_te, X_tr, y_tr,
                                  best_p["K"], s, s, s, s)
            pred = wsvm(C, X_tr, y_tr, X_te, best_w)
            sp, se, gm = Gmean(y_te, pred)
            f1  = f1_score(y_te, pred, pos_label=1, zero_division=0)
            acc = accuracy_score(y_te, pred)
            auc = roc_auc_score(y_te, pred)
            append_csv_row(csv_path, make_row(
                rep, fold, round(time.time()-t0, 4),
                "GridSearch-AFW-CIL", f"K={best_p['K']},s={best_p['sigma']}",
                sp, se, gm, f1, acc, auc,
                str(confusion_matrix(y_te, pred, labels=[-1., 1.]).tolist())))

            # ── 4. PSO-AFW-CIL ───────────────────────────────────
            t0 = time.time()
            pso = PSO_AFWCIL(
                PSO_PARTICLES, PSO_ITERS, BOUNDS, C, T_INNER,
                X_tr, y_tr, X_te, y_te, NAMEMETHOD, NAMEFUNCTION)
            gbest_pos, gbest_val, _ = pso.optimize()
            K_b = int(round(float(gbest_pos[0])))
            _, posX, negX = is_tomek(X_tr, y_tr, class_type=[-1.0])
            w_init = fuzzy_weight(0.5, 0.8, 0.2, X_tr, y_tr, NAMEMETHOD, NAMEFUNCTION)
            best_w, _ = lfb_pso(C, posX, negX, w_init, T_INNER,
                                  X_te, y_te, X_tr, y_tr,
                                  K_b, gbest_pos[1], gbest_pos[2],
                                  gbest_pos[3], gbest_pos[4])
            pred = wsvm(C, X_tr, y_tr, X_te, best_w)
            sp, se, gm = Gmean(y_te, pred)
            f1  = f1_score(y_te, pred, pos_label=1, zero_division=0)
            acc = accuracy_score(y_te, pred)
            auc = roc_auc_score(y_te, pred)
            fn_str = (f"K={K_b},s1={gbest_pos[1]:.3f},s2={gbest_pos[2]:.3f},"
                      f"s3={gbest_pos[3]:.3f},s4={gbest_pos[4]:.3f}")
            append_csv_row(csv_path, make_row(
                rep, fold, round(time.time()-t0, 4),
                "PSO-AFW-CIL", fn_str,
                sp, se, gm, f1, acc, auc,
                str(confusion_matrix(y_te, pred, labels=[-1., 1.]).tolist())))

            print(f"  Rep {rep:02d} | Fold {fold} | PSO Gmean={gbest_val:.3f}")

    # ── Tóm tắt KB1 ──────────────────────────────────────────
    df_kb1 = pd.read_csv(csv_path)
    for col in ['SP', 'SE', 'Gmean', 'F1 Score', 'Accuracy', 'AUC']:
        df_kb1[col] = pd.to_numeric(df_kb1[col], errors='coerce')
    agg = {}
    for m in ['Gmean', 'SE', 'SP', 'F1 Score', 'AUC']:
        agg[f"{m}_mean"] = (m, 'mean')
        agg[f"{m}_std"]  = (m, 'std')
    df_sum1 = df_kb1.groupby("Name Method").agg(**agg).reset_index()

    # Lưu file tổng hợp
    sum_path = os.path.join(run_dir, f"KB1_summary_{ts}.csv")
    df_sum1.to_csv(sum_path, index=False)

    print(f"\n✔ KB1 xong. CSV: {csv_path}")
    print(f"✔ Tổng hợp: {sum_path}")
    print(df_sum1[["Name Method", "Gmean_mean", "Gmean_std", "SE_mean"]].to_string(index=False))
    return csv_path, df_sum1


# ── Chạy KB1 ─────────────────────────────────────────────────
csv_kb1, df_kb1_sum = run_kb1(X_all, y_all, RUN_DIR)


---
## Kịch bản 2 (KB2) — Pipeline Hybrid 6 Mô hình

| # | Phương pháp |
|---|---|
| 1 | W-SVM |
| 2 | BL-SMOTE + W-SVM |
| 3 | AFW-CIL (σ cố định) |
| 4 | PSO-AFW-CIL |
| 5 | BL-SMOTE + AFW-CIL |
| 6 | **BL-SMOTE + PSO-AFW-CIL** ← đề xuất |

CSV + TXT log lưu theo từng repeat vào `{RUN_DIR}/KB2_*`

In [ ]:
def run_kb2(X_all, y_all, run_dir, dataset_name=DATASET_NAME):
    """
    KB2: 6-cấu hình hybrid pipeline với BL-SMOTE.
    Lưu CSV per-fold + TXT log per-repeat.
    Trả về: (csv_path, txt_path, df_summary)
    """
    ts       = datetime.now().strftime("%d%m%Y_%H%M%S")
    csv_path = os.path.join(run_dir, f"KB2_{dataset_name}_{ts}.csv")
    txt_path = os.path.join(run_dir, "logs", f"KB2_{dataset_name}_{ts}.txt")

    print(f"\n{'='*60}")
    print(f"KB2 | Dataset: {dataset_name} | {N_REPEATS}×{N_SPLITS}-fold CV")
    print(f"CSV: {csv_path}")
    print(f"{'='*60}")

    with open(txt_path, 'w', encoding='utf-8') as logf:
        logf.write(f"KB2 | Dataset: {dataset_name}\n")
        logf.write(f"Bắt đầu: {datetime.now()}\n\n")

        for rep in range(1, N_REPEATS + 1):
            skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=rep * 7)
            for fold, (tr_idx, te_idx) in enumerate(skf.split(X_all, y_all), 1):
                X_tr_raw, X_te_raw = X_all[tr_idx], X_all[te_idx]
                y_tr, y_te         = y_all[tr_idx], y_all[te_idx]

                sc   = StandardScaler()
                X_tr = sc.fit_transform(X_tr_raw)
                X_te = sc.transform(X_te_raw)

                sm = BorderlineSMOTE(random_state=rep)
                X_tr_sm, y_tr_sm = sm.fit_resample(X_tr, y_tr)[:2]

                logf.write(f"\n--- Rep {rep:02d} | Fold {fold} ---\n")
                logf.write(f"  Train={len(y_tr)} → SMOTE={len(y_tr_sm)}\n")

                def _record(method_name, fn_name, X_use, y_use, w):
                    t0   = time.time()
                    pred = wsvm(C, X_use, y_use, X_te, w)
                    sp, se, gm = Gmean(y_te, pred)
                    f1  = f1_score(y_te, pred, pos_label=1, zero_division=0)
                    acc = accuracy_score(y_te, pred)
                    auc = roc_auc_score(y_te, pred)
                    elapsed = round(time.time() - t0, 4)
                    cm_str  = str(confusion_matrix(y_te, pred, labels=[-1., 1.]).tolist())
                    append_csv_row(csv_path, make_row(
                        rep, fold, elapsed, method_name, fn_name,
                        sp, se, gm, f1, acc, auc, cm_str))
                    logf.write(f"  {method_name:30s} | Gm={gm:.4f} | AUC={auc:.4f} | t={elapsed}s\n")
                    return gm

                # 1. W-SVM
                _record("W-SVM", "baseline", X_tr, y_tr, np.ones(len(y_tr)))

                # 2. BL-SMOTE + W-SVM
                _record("BL-SMOTE+W-SVM", "borderline_smote",
                        X_tr_sm, y_tr_sm, np.ones(len(y_tr_sm)))

                # 3. AFW-CIL (fixed σ)
                _, posX, negX = is_tomek(X_tr, y_tr, class_type=[-1.0])
                w_init = fuzzy_weight(0.5, 0.8, 0.2, X_tr, y_tr, NAMEMETHOD, NAMEFUNCTION)
                best_w, _, _ = lfb_fixed(C, posX, negX, w_init, T_INNER, X_te, y_te, X_tr, y_tr)
                _record("AFW-CIL_fixed", "fixed_sigma_1.2", X_tr, y_tr, best_w)

                # 4. PSO-AFW-CIL
                pso = PSO_AFWCIL(PSO_PARTICLES, PSO_ITERS, BOUNDS, C, T_INNER,
                                  X_tr, y_tr, X_te, y_te, NAMEMETHOD, NAMEFUNCTION)
                gbpos, gbval, _ = pso.optimize()
                K_b = int(round(float(gbpos[0])))
                _, posX, negX = is_tomek(X_tr, y_tr, class_type=[-1.0])
                w_init = fuzzy_weight(0.5, 0.8, 0.2, X_tr, y_tr, NAMEMETHOD, NAMEFUNCTION)
                best_w, _ = lfb_pso(C, posX, negX, w_init, T_INNER, X_te, y_te, X_tr, y_tr,
                                     K_b, gbpos[1], gbpos[2], gbpos[3], gbpos[4])
                fn_str = (f"K={K_b},s1={gbpos[1]:.3f},s2={gbpos[2]:.3f},"
                          f"s3={gbpos[3]:.3f},s4={gbpos[4]:.3f}")
                _record("PSO-AFW-CIL", fn_str, X_tr, y_tr, best_w)

                # 5. BL-SMOTE + AFW-CIL (fixed)
                _, posX2, negX2 = is_tomek(X_tr_sm, y_tr_sm, class_type=[-1.0])
                w_init2 = fuzzy_weight(0.5, 0.8, 0.2, X_tr_sm, y_tr_sm, NAMEMETHOD, NAMEFUNCTION)
                best_w2, _, _ = lfb_fixed(C, posX2, negX2, w_init2, T_INNER,
                                           X_te, y_te, X_tr_sm, y_tr_sm)
                _record("BL-SMOTE+AFW-CIL_fixed", "fixed_sigma_1.2",
                        X_tr_sm, y_tr_sm, best_w2)

                # 6. BL-SMOTE + PSO-AFW-CIL (mô hình đề xuất)
                pso2 = PSO_AFWCIL(PSO_PARTICLES, PSO_ITERS, BOUNDS, C, T_INNER,
                                   X_tr_sm, y_tr_sm, X_te, y_te, NAMEMETHOD, NAMEFUNCTION)
                gbpos2, gbval2, _ = pso2.optimize()
                K_b2 = int(round(float(gbpos2[0])))
                _, posX2, negX2 = is_tomek(X_tr_sm, y_tr_sm, class_type=[-1.0])
                w_init2 = fuzzy_weight(0.5, 0.8, 0.2, X_tr_sm, y_tr_sm, NAMEMETHOD, NAMEFUNCTION)
                best_w2, _ = lfb_pso(C, posX2, negX2, w_init2, T_INNER,
                                      X_te, y_te, X_tr_sm, y_tr_sm,
                                      K_b2, gbpos2[1], gbpos2[2], gbpos2[3], gbpos2[4])
                fn_str2 = (f"K={K_b2},s1={gbpos2[1]:.3f},s2={gbpos2[2]:.3f},"
                           f"s3={gbpos2[3]:.3f},s4={gbpos2[4]:.3f}")
                _record("BL-SMOTE+PSO-AFW-CIL", fn_str2, X_tr_sm, y_tr_sm, best_w2)

                print(f"  Rep {rep:02d} | Fold {fold} | PSO={gbval:.3f} | SMOTE+PSO={gbval2:.3f}")

            # ── Checkpoint sau mỗi repeat ──────────────────────
            ckpt_path = os.path.join(run_dir,
                f"KB2_{dataset_name}_rep{rep:02d}_checkpoint.csv")
            if os.path.exists(csv_path):
                import shutil
                shutil.copy(csv_path, ckpt_path)
            logf.write(f"\n[✔] Checkpoint rep {rep}: {ckpt_path}\n")

        logf.write(f"\nKết thúc: {datetime.now()}\n")

    # ── Tóm tắt KB2 ──────────────────────────────────────────
    df_kb2 = pd.read_csv(csv_path)
    for col in ['Gmean', 'SE', 'SP', 'F1 Score', 'AUC']:
        df_kb2[col] = pd.to_numeric(df_kb2[col], errors='coerce')
    agg = {}
    for m in ['Gmean', 'SE', 'SP', 'F1 Score', 'AUC']:
        agg[f"{m}_mean"] = (m, 'mean')
        agg[f"{m}_std"]  = (m, 'std')
    df_sum2 = df_kb2.groupby("Name Method").agg(**agg).reset_index()

    sum_path = os.path.join(run_dir, f"KB2_summary_{ts}.csv")
    df_sum2.to_csv(sum_path, index=False)

    print(f"\n✔ KB2 xong. CSV: {csv_path} | TXT: {txt_path}")
    print(f"✔ Tổng hợp: {sum_path}")
    print(df_sum2[["Name Method", "Gmean_mean", "Gmean_std"]].to_string(index=False))
    return csv_path, txt_path, df_sum2


# ── Chạy KB2 ─────────────────────────────────────────────────
csv_kb2, txt_kb2, df_kb2_sum = run_kb2(X_all, y_all, RUN_DIR)


---
## Kịch bản 3 (KB3) — Stress Test: IR Variants bằng `change_rate_data`

Tạo các biến thể của **haberman** có IR giảm dần:

| Target IR (pos%) | Mẫu dương ≈ | Ghi chú |
|---|---|---|
| 20% | 61 | phải < 26.5% tự nhiên |
| 15% | 46 | |
| 10% | 31 | |
| 8%  | 24 | |
| 6%  | 18 | |
| 4%  | 12 | |
| 2%  | 6  | rất hiếm |

Dùng `change_rate_data(X, y, new_rate)` để lấy mẫu con lớp dương.  
CSV riêng per-IR + biểu đồ G-mean vs IR lưu vào `{RUN_DIR}/`

In [ ]:
# ── Các mức IR cần thử nghiệm ────────────────────────────────
# Haberman: 81 pos / 306 total = 26.5% → chỉ giảm được ≤ 26%
KB3_IR_LIST = [0.20, 0.15, 0.10, 0.08, 0.06, 0.04, 0.02]

def run_kb3(X_all, y_all, run_dir, dataset_name=DATASET_NAME,
            ir_list=None):
    """
    KB3: Stress-test trên các biến thể IR của dataset gốc.
    change_rate_data() tạo bộ dữ liệu con với IR mong muốn.
    So sánh W-SVM vs PSO-AFW-CIL.
    Trả về: (df_summary, csv_paths_dict)
    """
    if ir_list is None:
        ir_list = KB3_IR_LIST

    ts         = datetime.now().strftime("%d%m%Y_%H%M%S")
    summary_rows = []
    csv_paths    = {}

    print(f"\n{'='*60}")
    print(f"KB3 | Dataset: {dataset_name} | IR list: {ir_list}")
    print(f"{'='*60}")

    for ir_target in ir_list:
        ir_pct = int(ir_target * 100)
        print(f"\n[KB3] IR = {ir_target*100:.0f}%  ({ir_target}) ...")

        # ── Tạo biến thể IR ────────────────────────────────
        try:
            X_ir, y_ir = change_rate_data(X_all, y_all, new_rate=ir_target)
        except Exception as e:
            print(f"  [!] change_rate_data lỗi: {e} → bỏ qua IR={ir_target}")
            continue

        pos_c = int(np.sum(y_ir == 1.0))
        neg_c = int(np.sum(y_ir == -1.0))
        actual_ir = pos_c / len(y_ir)
        print(f"  Sau change_rate: {len(y_ir)} mẫu | +1={pos_c} | -1={neg_c} | "
              f"IR_thực={actual_ir:.4f}")

        # ── Lưu checkpoint biến thể này ────────────────────
        df_ir = pd.DataFrame(X_ir)
        df_ir['label']     = y_ir
        ckpt_ir = os.path.join(run_dir, f"KB3_{dataset_name}_IR{ir_pct}pct_data.csv")
        df_ir.to_csv(ckpt_ir, index=False)

        csv_path = os.path.join(run_dir, f"KB3_{dataset_name}_IR{ir_pct}pct_{ts}.csv")
        csv_paths[ir_target] = csv_path

        # Kiểm tra đủ mẫu cho StratifiedKFold
        min_class = min(pos_c, neg_c)
        n_splits_actual = min(N_SPLITS, min_class)
        if n_splits_actual < 2:
            print(f"  [!] Quá ít mẫu lớp thiểu số ({min_class}) → bỏ qua")
            continue

        wsvm_gm_list, pso_gm_list = [], []

        for rep in range(1, N_REPEATS + 1):
            skf = StratifiedKFold(n_splits=n_splits_actual,
                                   shuffle=True, random_state=rep * 7)
            for fold, (tr_idx, te_idx) in enumerate(skf.split(X_ir, y_ir), 1):
                X_tr_raw, X_te_raw = X_ir[tr_idx], X_ir[te_idx]
                y_tr, y_te         = y_ir[tr_idx], y_ir[te_idx]

                sc   = StandardScaler()
                X_tr = sc.fit_transform(X_tr_raw)
                X_te = sc.transform(X_te_raw)

                # W-SVM baseline
                pred_b = wsvm(C, X_tr, y_tr, X_te, np.ones(len(y_tr)))
                _, _, gm_b = Gmean(y_te, pred_b)
                wsvm_gm_list.append(gm_b)
                sp_b, se_b, gm_b2 = Gmean(y_te, pred_b)
                f1_b  = f1_score(y_te, pred_b, pos_label=1, zero_division=0)
                acc_b = accuracy_score(y_te, pred_b)
                auc_b = roc_auc_score(y_te, pred_b)
                append_csv_row(csv_path, make_row(
                    rep, fold, 0.0, "W-SVM", f"IR={ir_target}",
                    sp_b, se_b, gm_b2, f1_b, acc_b, auc_b,
                    str(confusion_matrix(y_te, pred_b, labels=[-1., 1.]).tolist())))

                # PSO-AFW-CIL
                pso = PSO_AFWCIL(PSO_PARTICLES, PSO_ITERS, BOUNDS, C, T_INNER,
                                  X_tr, y_tr, X_te, y_te, NAMEMETHOD, NAMEFUNCTION)
                gbpos, gbval, _ = pso.optimize()
                K_b = int(round(float(gbpos[0])))
                _, posX, negX = is_tomek(X_tr, y_tr, class_type=[-1.0])
                w_init = fuzzy_weight(0.5, 0.8, 0.2, X_tr, y_tr, NAMEMETHOD, NAMEFUNCTION)
                best_w, _ = lfb_pso(C, posX, negX, w_init, T_INNER,
                                     X_te, y_te, X_tr, y_tr,
                                     K_b, gbpos[1], gbpos[2], gbpos[3], gbpos[4])
                pred_p = wsvm(C, X_tr, y_tr, X_te, best_w)
                _, _, gm_p = Gmean(y_te, pred_p)
                pso_gm_list.append(gm_p)
                sp_p, se_p, gm_p2 = Gmean(y_te, pred_p)
                f1_p  = f1_score(y_te, pred_p, pos_label=1, zero_division=0)
                acc_p = accuracy_score(y_te, pred_p)
                auc_p = roc_auc_score(y_te, pred_p)
                fn_str = (f"K={K_b},s1={gbpos[1]:.3f},s2={gbpos[2]:.3f},"
                          f"s3={gbpos[3]:.3f},s4={gbpos[4]:.3f}")
                append_csv_row(csv_path, make_row(
                    rep, fold, 0.0, "PSO-AFW-CIL", fn_str,
                    sp_p, se_p, gm_p2, f1_p, acc_p, auc_p,
                    str(confusion_matrix(y_te, pred_p, labels=[-1., 1.]).tolist())))

            print(f"  IR={ir_target*100:.0f}% | Rep {rep:02d} ✓")

        summary_rows.append({
            "IR_target":   ir_target,
            "IR_actual":   round(actual_ir, 4),
            "n_samples":   len(y_ir),
            "pos_count":   pos_c,
            "WSVM_Gm_mean": round(np.mean(wsvm_gm_list), 4),
            "WSVM_Gm_std":  round(np.std(wsvm_gm_list),  4),
            "PSO_Gm_mean":  round(np.mean(pso_gm_list),  4),
            "PSO_Gm_std":   round(np.std(pso_gm_list),   4),
        })

    # ── Tổng hợp KB3 ─────────────────────────────────────────
    df_sum3 = pd.DataFrame(summary_rows)

    sum_path = os.path.join(run_dir, f"KB3_summary_{ts}.csv")
    df_sum3.to_csv(sum_path, index=False)

    print(f"\n{'='*60}")
    print("BẢNG TỔNG KẾT KB3 (G-mean mean ± std)")
    print(df_sum3.to_string(index=False))

    # ── Biểu đồ G-mean vs IR ─────────────────────────────────
    if len(df_sum3) > 0:
        _, ax = plt.subplots(figsize=(9, 5))
        ax.errorbar(df_sum3["IR_actual"], df_sum3["WSVM_Gm_mean"],
                    yerr=df_sum3["WSVM_Gm_std"], marker='o',
                    label='W-SVM', capsize=4, linestyle='--')
        ax.errorbar(df_sum3["IR_actual"], df_sum3["PSO_Gm_mean"],
                    yerr=df_sum3["PSO_Gm_std"], marker='s',
                    label='PSO-AFW-CIL', capsize=4)
        ax.set_xlabel("Imbalance Ratio – tỷ lệ lớp dương (pos/total)")
        ax.set_ylabel("G-mean (mean ± std)")
        ax.set_title(f"KB3: G-mean vs Imbalance Ratio — {dataset_name}")
        ax.legend()
        ax.grid(True, linestyle='--', alpha=0.6)
        ax.invert_xaxis()   # từ cao → thấp (stress tăng dần)
        plt.tight_layout()
        plot_path = os.path.join(run_dir, f"KB3_gmean_vs_ir_{ts}.png")
        plt.savefig(plot_path, dpi=150)
        plt.show()
        print(f"✔ Biểu đồ: {plot_path}")

    print(f"✔ KB3 xong. Tổng hợp: {sum_path}")
    return df_sum3, csv_paths


# ── Chạy KB3 ─────────────────────────────────────────────────
df_kb3_sum, csv_kb3_dict = run_kb3(X_all, y_all, RUN_DIR)


---
## Tổng hợp Kết quả & Trực quan hoá

In [ ]:
def compute_average_result(csv_path, metrics=None):
    """Đọc CSV → tính mean±std theo 'Name Method'."""
    if metrics is None:
        metrics = ['SP', 'SE', 'Gmean', 'F1 Score', 'Accuracy', 'AUC']
    df = pd.read_csv(csv_path)
    for m in metrics:
        df[m] = pd.to_numeric(df[m], errors='coerce')
    agg = {}
    for m in metrics:
        agg[f"{m}_mean"] = (m, 'mean')
        agg[f"{m}_std"]  = (m, 'std')
    return df.groupby("Name Method").agg(**agg).reset_index()


def plot_comparison(summary_df, metric="Gmean",
                    title="So sánh phương pháp", save_path=None):
    """Biểu đồ cột mean ± std cho từng phương pháp."""
    mean_col = f"{metric}_mean"
    std_col  = f"{metric}_std"
    _, ax = plt.subplots(figsize=(max(8, len(summary_df) * 1.5), 5))
    colors = plt.colormaps['tab10'](np.linspace(0, 1, len(summary_df)))
    ax.bar(summary_df["Name Method"], summary_df[mean_col],
           yerr=summary_df[std_col],
           color=colors, alpha=0.85, capsize=5, edgecolor='black')
    ax.set_ylabel(f"{metric} (mean ± std)")
    ax.set_title(title)
    ax.tick_params(axis='x', rotation=30)
    ax.grid(axis='y', linestyle='--', alpha=0.5)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150)
        print(f"✔ Plot saved: {save_path}")
    plt.show()


# ── Tổng hợp từ tất cả CSV của lần chạy này ──────────────────
all_csvs = sorted(glob.glob(os.path.join(RUN_DIR, "KB*.csv")))
# Lọc bỏ checkpoint & summary
scenario_csvs = [f for f in all_csvs
                 if "checkpoint" not in f and "summary" not in f and "data" not in f]

print(f"\nThư mục: {RUN_DIR}")
print(f"Files CSV kịch bản:")
for f in scenario_csvs:
    print(f"  {os.path.basename(f)}")

# ── Vẽ biểu đồ cho từng kịch bản ────────────────────────────
for csv_f in scenario_csvs:
    name = os.path.basename(csv_f).replace(".csv", "")
    try:
        s = compute_average_result(csv_f)
        print(f"\n{'='*55}")
        print(f"Tổng kết: {name}")
        display(s[["Name Method", "Gmean_mean", "Gmean_std", "AUC_mean"]])
        for metric in ["Gmean", "AUC", "F1 Score"]:
            save_p = os.path.join(RUN_DIR, f"plot_{name}_{metric.replace(' ','_')}.png")
            plot_comparison(s, metric=metric,
                            title=f"{name} | {metric}",
                            save_path=save_p)
    except Exception as e:
        print(f"[!] Không thể vẽ {name}: {e}")

# ── Hiển thị tổng kết KB3 ────────────────────────────────────
if 'df_kb3_sum' in dir() and len(df_kb3_sum) > 0:
    print("\n\nKB3 — STRESS TEST SUMMARY:")
    display(df_kb3_sum)

print(f"\n✔ Tất cả kết quả trong: {RUN_DIR}")
print("Contents:")
for f in sorted(os.listdir(RUN_DIR)):
    fpath = os.path.join(RUN_DIR, f)
    size  = os.path.getsize(fpath) if os.path.isfile(fpath) else 0
    print(f"  {f:<60}  {size:>8} bytes")
